# Rerankers — Types, Usage & When to Use

## What is a Reranker?

```
WITHOUT Reranker:
  Query → VectorDB → Top-5 chunks (by embedding similarity) → LLM
  Problem: embedding similarity ≠ actual relevance. Wrong chunks go to LLM.

WITH Reranker:
  Query → VectorDB → Top-20 chunks → Reranker scores each → Top-5 best → LLM
  Reranker reads BOTH query + chunk together → much more accurate scoring
```

## The 4 Types of Rerankers

| Type | Speed | Accuracy | Cost | Best For |
|---|---|---|---|---|
| **Cross-Encoder** | Medium | ⭐⭐⭐⭐ | Free (local) | General RAG, English |
| **Bi-Encoder** | Fast | ⭐⭐⭐ | Free (local) | Large candidate sets |
| **LLM-Based** | Slow | ⭐⭐⭐⭐⭐ | High (API calls) | Medical, Legal, Financial |
| **Cohere API** | Fast | ⭐⭐⭐⭐⭐ | Paid API | Production, multi-language |

## How Reranking Fits in RAG Pipeline

```
User Query
    ↓
VectorDB (Bi-Encoder embeddings)
    ↓  retrieve top-20 candidates  (cast WIDE net)
Reranker (Cross-Encoder)
    ↓  score all 20 against query  (precise scoring)
    ↓  keep only top-3 best docs
LLM
    ↓  gets only HIGHEST QUALITY context
Final Answer  ← much more accurate
```

## Cross-Encoder vs Bi-Encoder — Key Difference

```
Bi-Encoder (VectorDB retrieval):
  Query  → embed → [0.23, -0.45, 0.87]   separate vectors
  Doc    → embed → [0.21, -0.43, 0.85]   compared by cosine similarity
  Fast but misses context between query and doc

Cross-Encoder (Reranker):
  [QUERY + DOC together] → model reads both → relevance score = 0.94
  Slower but understands the RELATIONSHIP between query and doc
```

## Step 1 — Install Dependencies

In [ ]:
!pip install sentence-transformers chromadb scikit-learn cohere langchain-openai

## Step 2 — Sample Documents (Medical Use Case)

These are the candidate documents retrieved from a VectorDB.  
The reranker's job is to score each one and find which is truly most relevant to the query.

In [ ]:
# These simulate what a VectorDB would return for a patient query
query = "What is John's diabetes medication and dosage?"

candidate_docs = [
    "John takes Metformin 500mg twice daily since 2024 for Type 2 Diabetes.",        # very relevant
    "Metformin is a biguanide class drug that reduces hepatic glucose production.",   # somewhat relevant
    "John's BP history: Jan 2024: 145/92, Jun 2024: 150/95, Dec 2024: 148/93.",     # weakly relevant
    "The weather in London is cloudy today with temperatures around 15°C.",           # not relevant
    "Diabetes affects 537 million adults worldwide according to IDF 2021.",           # weakly relevant
    "John current medications: Amlodipine 5mg (since Jan 2026), Metformin 500mg.",   # very relevant
    "Insulin therapy is considered when HbA1c remains above 9% despite oral agents.", # somewhat relevant
    "John HbA1c trend: Jan 2024: 8.2%, Jun 2024: 7.9%, Dec 2024: 8.1%.",            # relevant
]

print(f"Query: '{query}'")
print(f"\nCandidate documents ({len(candidate_docs)}):")
for i, doc in enumerate(candidate_docs):
    print(f"  [{i}] {doc[:70]}...")

## Type 1 — Cross-Encoder Reranker (Most Popular)

Reads query + document **together** in one pass — understands the relationship between them.

```
Input:  [SEP] "What is John's medication?" [SEP] "John takes Metformin 500mg..."
Output: score = 0.94  ← very relevant

Input:  [SEP] "What is John's medication?" [SEP] "Weather in London is cloudy..."
Output: score = 0.01  ← not relevant at all
```

**Best model**: `cross-encoder/ms-marco-MiniLM-L-6-v2` — 22MB, fast, very accurate

In [ ]:
from sentence_transformers import CrossEncoder

# Load cross-encoder model (downloads ~22MB on first run)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Create (query, doc) pairs — model reads both together
pairs = [(query, doc) for doc in candidate_docs]

# Score each pair
scores = cross_encoder.predict(pairs)

# Sort by score descending
ranked = sorted(zip(scores, candidate_docs), reverse=True)

print("=" * 65)
print("CROSS-ENCODER RERANKER RESULTS")
print("=" * 65)
for i, (score, doc) in enumerate(ranked):
    marker = "✅" if i < 3 else "❌"
    print(f"{marker} [{i+1}] Score: {score:.3f}")
    print(f"      {doc[:70]}...")
    print()

print(f"\nTop 3 docs sent to LLM (best quality context):")

## Type 2 — Bi-Encoder Reranker (Faster)

Encodes query and documents **separately**, then computes cosine similarity.

```
Query  → SentenceTransformer → vector [0.23, -0.45, 0.87, ...]
Doc1   → SentenceTransformer → vector [0.21, -0.43, 0.85, ...]
cos_sim(query_vec, doc1_vec) = 0.92  ← high similarity

Doc2   → SentenceTransformer → vector [0.80, 0.20, -0.10, ...]
cos_sim(query_vec, doc2_vec) = 0.15  ← low similarity
```

Faster than Cross-Encoder because docs can be pre-encoded and cached.  
**Model**: `BAAI/bge-reranker-base` — optimized specifically for reranking task

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load bi-encoder model (optimized for reranking)
bi_encoder = SentenceTransformer("BAAI/bge-reranker-base")

# Encode query and all documents separately
query_embedding = bi_encoder.encode([query])              # shape: (1, 768)
doc_embeddings  = bi_encoder.encode(candidate_docs)       # shape: (8, 768)

# Compute cosine similarity between query and each doc
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]  # (8,)

# Sort by similarity descending
ranked_bi = sorted(zip(similarities, candidate_docs), reverse=True)

print("=" * 65)
print("BI-ENCODER RERANKER RESULTS")
print("=" * 65)
for i, (score, doc) in enumerate(ranked_bi):
    marker = "✅" if i < 3 else "❌"
    print(f"{marker} [{i+1}] Score: {score:.4f}")
    print(f"      {doc[:70]}...")
    print()

print("\nNote: Bi-Encoder is faster (docs can be pre-cached) but slightly less accurate than Cross-Encoder")

## Type 3 — LLM-Based Reranker (Most Intelligent)

Uses a language model (GPT-4o, Claude, Gemini) to **reason about** document relevance.

```
Prompt → LLM:
  "Rate how relevant this document is for answering the query.
   Query: 'What is John's diabetes medication?'
   Document: 'John takes Metformin 500mg twice daily...'
   Return a score from 0.0 (not relevant) to 1.0 (perfectly relevant)"

LLM response: 0.97  ← understands John's name, medication type, dosage
```

**When to use**: Medical, legal, financial domains where precision is critical.  
**Trade-off**: Slow (1 API call per doc) + expensive (GPT-4 tokens), but most accurate.

In [ ]:
import os
import re
from langchain_openai import ChatOpenAI

# Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "sk-..."  # uncomment and set your key

def llm_score_document(llm, query: str, document: str) -> float:
    """Ask LLM to score relevance of a document for a query (0.0 - 1.0)."""
    prompt = f"""You are a relevance scoring assistant.
Rate how relevant the document is for answering the query.

Query: "{query}"
Document: "{document}"

Return ONLY a decimal number between 0.0 (completely irrelevant) and 1.0 (perfectly relevant).
No explanation, just the number."""
    
    response = llm.invoke(prompt)
    try:
        # Extract float from response
        score = float(re.search(r"[0-9]+\.?[0-9]*", response.content).group())
        return min(max(score, 0.0), 1.0)  # clamp to [0, 1]
    except:
        return 0.0

def llm_reranker(query: str, documents: list, top_k: int = 3) -> list:
    """Rerank documents using LLM scoring."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    scored = []
    for doc in documents:
        score = llm_score_document(llm, query, doc)
        scored.append((score, doc))
    
    # Sort by LLM score descending
    scored.sort(reverse=True)
    return scored[:top_k]

# Example usage (requires OPENAI_API_KEY):
# top_docs = llm_reranker(query, candidate_docs, top_k=3)
# for score, doc in top_docs:
#     print(f"Score: {score:.2f} | {doc[:70]}...")

# Simulated output for demonstration:
simulated_llm_scores = [
    (0.97, candidate_docs[0]),  # "John takes Metformin 500mg..."
    (0.92, candidate_docs[5]),  # "John current medications..."
    (0.85, candidate_docs[7]),  # "John HbA1c trend..."
    (0.62, candidate_docs[1]),  # "Metformin is a biguanide..."
    (0.38, candidate_docs[6]),  # "Insulin therapy is considered..."
    (0.22, candidate_docs[2]),  # "John's BP history..."
    (0.10, candidate_docs[4]),  # "Diabetes affects 537 million..."
    (0.01, candidate_docs[3]),  # "Weather in London..."
]
print("=" * 65)
print("LLM-BASED RERANKER RESULTS (simulated)")
print("=" * 65)
for i, (score, doc) in enumerate(simulated_llm_scores):
    marker = "✅" if i < 3 else "❌"
    print(f"{marker} [{i+1}] LLM Score: {score:.2f}")
    print(f"      {doc[:70]}...")
    print()
print("→ LLM understands patient context (John's name, medication purpose, etc.)")

## Type 4 — Cohere Reranker (Production-Ready API)

Cohere's hosted reranking API — no local model needed, multi-language support.

```python
co.rerank(
    model    = "rerank-english-v3.0",   # or rerank-multilingual-v3.0
    query    = "John's diabetes medication",
    documents= candidate_docs,
    top_n    = 3
)
# Returns: ranked results with relevance_score for each doc
```

**Advantages**:
- No GPU / no local model — just an API call
- Handles multi-language (rerank-multilingual-v3.0)
- Used in production by many enterprises
- 5000 free API calls/month on trial plan

In [ ]:
import cohere

# Set your Cohere API key
# COHERE_API_KEY = "your-key-here"  # get from https://dashboard.cohere.com

def cohere_reranker(query: str, documents: list, top_n: int = 3) -> list:
    """Rerank documents using Cohere API."""
    co = cohere.Client(os.environ.get("COHERE_API_KEY", "your-key-here"))
    
    results = co.rerank(
        model     = "rerank-english-v3.0",   # or "rerank-multilingual-v3.0"
        query     = query,
        documents = documents,
        top_n     = top_n,
    )
    
    ranked_docs = []
    for r in results.results:
        ranked_docs.append({
            "rank"            : r.index + 1,
            "relevance_score" : r.relevance_score,
            "document"        : documents[r.index]
        })
    return ranked_docs

# Example usage (requires COHERE_API_KEY):
# top_docs = cohere_reranker(query, candidate_docs, top_n=3)
# for doc in top_docs:
#     print(f"Score: {doc['relevance_score']:.4f} | {doc['document'][:60]}...")

# Simulated output for demonstration:
print("=" * 65)
print("COHERE RERANKER API RESULTS (simulated)")
print("=" * 65)
print(f"API Call: co.rerank(model='rerank-english-v3.0', top_n=3)")
print()

simulated_cohere = [
    {"rank": 1, "relevance_score": 0.9978, "document": candidate_docs[0]},
    {"rank": 2, "relevance_score": 0.9821, "document": candidate_docs[5]},
    {"rank": 3, "relevance_score": 0.8534, "document": candidate_docs[7]},
]
for doc in simulated_cohere:
    print(f"✅ Rank {doc['rank']} | Score: {doc['relevance_score']:.4f}")
    print(f"   {doc['document'][:70]}...")
    print()

print("→ Cohere API is production-grade, handles scale without local GPU")

## Full RAG Pipeline With Reranker

```
User Query
    ↓
[Stage 1] VectorDB retrieval   → cast WIDE net (top-20)
    ↓
[Stage 2] Reranker scoring     → precise relevance scoring  
    ↓
[Stage 3] Keep only top-3      → quality over quantity
    ↓
[Stage 4] LLM generates answer → best possible context
```

Key idea: retrieve MORE candidates than you need, then ruthlessly filter to BEST quality.

In [ ]:
import chromadb
from sentence_transformers import CrossEncoder, SentenceTransformer
from langchain_openai import ChatOpenAI

# ─── Setup: VectorDB with sample medical knowledge ──────────────────────────
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="medical_records")

# Populate VectorDB with sample medical documents
all_docs = [
    "John takes Metformin 500mg twice daily since 2024 for Type 2 Diabetes.",
    "Metformin is a biguanide class drug that reduces hepatic glucose production.",
    "John's BP history: Jan 2024: 145/92, Jun 2024: 150/95, Dec 2024: 148/93.",
    "The weather in London is cloudy today with temperatures around 15°C.",
    "Diabetes affects 537 million adults worldwide according to IDF 2021.",
    "John current medications: Amlodipine 5mg (since Jan 2026), Metformin 500mg.",
    "Insulin therapy is considered when HbA1c remains above 9% despite oral agents.",
    "John HbA1c trend: Jan 2024: 8.2%, Jun 2024: 7.9%, Dec 2024: 8.1%.",
    "Type 2 Diabetes treatment guidelines recommend lifestyle changes first.",
    "John's weight: 82kg (Jan 2024) → 79kg (Jun 2024) → 77kg (Dec 2024).",
    "Metformin should be taken with meals to reduce gastrointestinal side effects.",
    "John's kidney function (eGFR): 78 mL/min — normal range for Metformin use.",
    "SGLT2 inhibitors are second-line after Metformin for Type 2 Diabetes.",
    "John reported mild nausea in first month of Metformin — resolved by month 2.",
    "HbA1c target for most Type 2 diabetic patients is less than 7.0%.",
]

# Add to ChromaDB
collection.add(
    documents=all_docs,
    ids=[f"doc_{i}" for i in range(len(all_docs))]
)

# ─── RAG Pipeline with Reranker ─────────────────────────────────────────────
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rag_with_reranker(query: str, top_k_retrieve: int = 10, top_k_final: int = 3) -> str:
    """
    Full RAG pipeline:
    1. VectorDB retrieval → top_k_retrieve candidates
    2. Cross-Encoder reranker → score all candidates  
    3. Keep top_k_final best documents
    4. LLM generates answer
    """
    # Stage 1: Wide retrieval from VectorDB
    results = collection.query(query_texts=[query], n_results=top_k_retrieve)
    retrieved_docs = results["documents"][0]
    print(f"[Stage 1] Retrieved {len(retrieved_docs)} candidates from VectorDB")
    
    # Stage 2: Rerank with Cross-Encoder
    pairs  = [(query, doc) for doc in retrieved_docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, retrieved_docs), reverse=True)
    print(f"[Stage 2] Reranker scored all {len(retrieved_docs)} docs")
    
    # Stage 3: Keep only top_k_final best docs
    top_docs = [doc for _, doc in ranked[:top_k_final]]
    print(f"[Stage 3] Kept top {top_k_final} highest-quality docs:")
    for i, (score, doc) in enumerate(ranked[:top_k_final]):
        print(f"          [{i+1}] score={score:.3f} | {doc[:60]}...")
    
    # Stage 4: LLM answers using only best context
    # Uncomment if you have OPENAI_API_KEY:
    # llm     = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    # context = "\n".join([f"[{i+1}] {d}" for i, d in enumerate(top_docs)])
    # prompt  = f"Based on this patient record context, answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    # answer  = llm.invoke(prompt).content
    # return answer
    
    # Simulated answer for demo:
    return f"[Simulated LLM Answer] John is prescribed Metformin 500mg twice daily for Type 2 Diabetes management."

# Run the pipeline
user_query = "What is John's diabetes medication and dosage?"
print("=" * 65)
print(f"QUERY: {user_query}")
print("=" * 65)
answer = rag_with_reranker(user_query, top_k_retrieve=10, top_k_final=3)
print(f"\n[Stage 4] LLM Answer: {answer}")

## Two-Stage Reranking (Advanced Pattern)

Used when you have a **very large** corpus (millions of documents).

```
Corpus: 1,000,000 documents
         ↓
[Stage 1] Bi-Encoder (fast)
         ↓  top-100 candidates in milliseconds
[Stage 2] Cross-Encoder (accurate)
         ↓  rerank 100 → keep top-5
[Stage 3] LLM
         ↓  answer with best 5 docs
```

**Why**: Cross-Encoder is slow on large sets. Bi-Encoder pre-filters to manageable size,  
then Cross-Encoder does precise scoring on the shortlist only.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Large corpus simulation (use candidate_docs from earlier as our "large corpus")
large_corpus = all_docs  # in real scenario this would be 10k+ docs

# ─── Two-Stage Reranker ──────────────────────────────────────────────────────
bi_encoder   = SentenceTransformer("BAAI/bge-reranker-base")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def two_stage_reranker(query: str, corpus: list, stage1_k: int = 10, stage2_k: int = 3) -> list:
    """
    Two-stage reranking:
    Stage 1: Bi-Encoder filters large corpus → stage1_k candidates (fast)
    Stage 2: Cross-Encoder reranks candidates → stage2_k final docs (accurate)
    """
    # STAGE 1: Bi-Encoder — fast filtering
    print(f"[Stage 1] Bi-Encoder: scoring {len(corpus)} docs...")
    query_emb = bi_encoder.encode([query])
    doc_embs  = bi_encoder.encode(corpus)
    sims      = cosine_similarity(query_emb, doc_embs)[0]
    
    # Keep top stage1_k candidates
    top_indices = np.argsort(sims)[::-1][:stage1_k]
    stage1_docs = [corpus[i] for i in top_indices]
    stage1_scores = [sims[i] for i in top_indices]
    print(f"           Filtered to top {stage1_k} candidates")
    
    # STAGE 2: Cross-Encoder — precise reranking of shortlist
    print(f"[Stage 2] Cross-Encoder: precisely scoring {stage1_k} candidates...")
    pairs  = [(query, doc) for doc in stage1_docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, stage1_docs), reverse=True)
    
    final_docs = ranked[:stage2_k]
    print(f"           Final top {stage2_k} documents:")
    for i, (score, doc) in enumerate(final_docs):
        print(f"           [{i+1}] score={score:.3f} | {doc[:65]}...")
    
    return [doc for _, doc in final_docs]

# Run two-stage pipeline
print("=" * 65)
print(f"TWO-STAGE RERANKING")
print(f"Query: '{query}'")
print("=" * 65)
final_docs = two_stage_reranker(
    query    = query,
    corpus   = large_corpus,
    stage1_k = 8,   # Bi-Encoder: keep 8 from corpus
    stage2_k = 3    # Cross-Encoder: keep best 3 from 8
)

print(f"\nFinal {len(final_docs)} docs ready for LLM:")
for i, doc in enumerate(final_docs):
    print(f"  [{i+1}] {doc}")

## Popular Reranker Models Reference

| Model | Type | Size | Speed | Accuracy | Notes |
|---|---|---|---|---|---|
| `cross-encoder/ms-marco-MiniLM-L-6-v2` | Cross-Encoder | 22MB | ⚡⚡⚡ | ⭐⭐⭐⭐ | Best default choice |
| `cross-encoder/ms-marco-MiniLM-L-12-v2` | Cross-Encoder | 33MB | ⚡⚡ | ⭐⭐⭐⭐⭐ | Better accuracy, slower |
| `BAAI/bge-reranker-base` | Cross-Encoder | 278MB | ⚡⚡ | ⭐⭐⭐⭐ | Good Chinese + English |
| `BAAI/bge-reranker-large` | Cross-Encoder | 560MB | ⚡ | ⭐⭐⭐⭐⭐ | Best local accuracy |
| `Cohere rerank-english-v3.0` | Hosted API | — | ⚡⚡⚡ | ⭐⭐⭐⭐⭐ | Production, English |
| `Cohere rerank-multilingual-v3.0` | Hosted API | — | ⚡⚡⚡ | ⭐⭐⭐⭐⭐ | 100+ languages |
| GPT-4o / Claude | LLM | — | ⚡ | ⭐⭐⭐⭐⭐ | Best for complex domains |

## When to Use Which Reranker?

| Scenario | Recommended Reranker | Why |
|---|---|---|
| General RAG (English) | `ms-marco-MiniLM-L-6-v2` | Fast, free, accurate |
| Medical / Legal / Financial | LLM-based (GPT-4o) | Needs domain reasoning |
| Production at scale | Cohere API | No GPU, multi-lang, SLA |
| Multilingual corpus | `bge-reranker-large` or Cohere | Both support multiple languages |
| Large corpus (>100k docs) | Two-stage (Bi→Cross) | Bi-Encoder pre-filters fast |
| Cost-sensitive project | `ms-marco-MiniLM-L-6-v2` | Completely free local model |
| Highest accuracy needed | `bge-reranker-large` | Largest local model |

## Summary

```
Reranker = Second opinion on your VectorDB results

Without reranker: hope top-K embedding matches are the most relevant
With reranker:    guarantee top-K sent to LLM are truly the most relevant

Rule of thumb:
  - Retrieve 3x-10x more than you need (e.g., need 3 → retrieve 20)
  - Let reranker score all candidates
  - Keep only the BEST N for LLM
  - Result: dramatically better RAG quality
```